# Projekt Grupowy — Predykcja Sprzedaży E-commerce (Olist)

## Cel projektu
Automatyczny system predykcji sprzedaży produktów dla e-commerce. System odpowiada na 3 pytania:
1. Co i w jakich ilościach się sprzeda?
2. Ile towaru trzymać w magazynie?
3. Które produkty zyskują na popularności, a które tracą?

## Zawartość notebooka
1. **Etap 0:** Pozyskanie danych (9 tabel) + łączenie wg schematu ERD
2. **Etap 1:** EDA — analiza rozkładów, sprzedaży w czasie, kategorii
3. **Etap 2:** Feature engineering — agregacja tygodniowa, lagi, rolling, momentum, święta
4. **Analiza korelacji** — cechy vs target, multicolinearność
5. **Etap 3:** 8 modeli: Baseline, Ridge, Random Forest, LightGBM, XGBoost, CatBoost, Prophet

## Wersje źródłowe (połączenie)
- Pełne łączenie 9 tabel + święta brazylijskie + target encoding
- Momentum, pct_change, rozszerzone lagi
- XGBoost, CatBoost
- Prophet (model szeregu czasowego)


## Etap 0: Pozyskanie danych

Łączymy 9 tabel zgodnie ze schematem ERD Olist (Kaggle).

**Kluczowa zasada:** Tabele o relacji 1:N (payments, reviews, geolocation) agregujemy 
PRZED mergem, inaczej "rozmnożymy" wiersze i zafałszujemy dane.

**Grain** (poziom szczegółowości) = 1 wiersz = 1 pozycja zamówienia (order_item).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Biblioteki załadowane ✓")


In [ ]:
# ── Wczytanie 9 tabel + polskich tłumaczeń ──
base_path = 'dane'

orders = pd.read_csv(f'{base_path}/olist_orders_dataset.csv')
order_items = pd.read_csv(f'{base_path}/olist_order_items_dataset.csv')
products = pd.read_csv(f'{base_path}/olist_products_dataset.csv')
sellers = pd.read_csv(f'{base_path}/olist_sellers_dataset.csv')
customers = pd.read_csv(f'{base_path}/olist_customers_dataset.csv')
payments = pd.read_csv(f'{base_path}/olist_order_payments_dataset.csv')
reviews = pd.read_csv(f'{base_path}/olist_order_reviews_dataset.csv')
geolocation = pd.read_csv(f'{base_path}/olist_geolocation_dataset.csv')
category_translation = pd.read_csv(f'{base_path}/product_category_name_translation.csv')

# ── Polskie nazwy produktów i kategorii ──
# Plik olist_products_named.csv zawiera dla KAŻDEGO product_id:
#   - product_name_pl     → polska nazwa produktu
#   - category_pl         → polska kategoria (74 unikalne wartości)
#   - category_group_pl   → polska nadkategoria (14 grup)
# Łączymy przez product_id (a NIE przez product_category_name), bo mapowanie
# jest na poziomie produktu — niektóre PT kategorie rozdzielają się na różne PL kategorie.
products_pl = pd.read_csv(f'{base_path}/olist_products_named.csv')
products_pl = products_pl[['product_id', 'product_name_pl', 'category_pl', 'category_group_pl']]

print("=== Kształty danych ===")
for name, tabel in zip(
    ['orders', 'order_items', 'products', 'sellers', 'customers',
     'payments', 'reviews', 'geolocation', 'category_translation', 'products_pl'],
    [orders, order_items, products, sellers, customers,
     payments, reviews, geolocation, category_translation, products_pl]
):
    print(f"{name:25s}: {tabel.shape}")

print(f"\nUnikalnych polskich kategorii: {products_pl['category_pl'].nunique()}")
print(f"Unikalnych polskich grup kategorii: {products_pl['category_group_pl'].nunique()}")

In [ ]:
# ── Agregacja tabel 1:N PRZED mergem ──
payments_agg = payments.groupby('order_id').agg(
    payment_value_total=('payment_value', 'sum'),
    payment_installments_max=('payment_installments', 'max'),
    payment_types_count=('payment_type', 'nunique'),
    payment_type_main=('payment_type', lambda x: x.mode().iloc[0])
).reset_index()

reviews_agg = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean'),
    review_count=('review_id', 'count')
).reset_index()

geo_dedup = geolocation.groupby('geolocation_zip_code_prefix').agg(
    customer_lat=('geolocation_lat', 'mean'),
    customer_lng=('geolocation_lng', 'mean')
).reset_index()

# ── Łączenie zgodnie ze schematem ERD ──
# Dodajemy merge z products_pl (po product_id) — daje nam:
#   - product_name_pl (polska nazwa konkretnego produktu)
#   - category_pl (polska kategoria — używana wszędzie dalej jako klucz)
#   - category_group_pl (polska nadkategoria)
df = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(category_translation, on='product_category_name', how='left')
    .merge(products_pl, on='product_id', how='left')           # ← polskie nazwy
    .merge(sellers, on='seller_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_agg, on='order_id', how='left')
    .merge(geo_dedup, left_on='customer_zip_code_prefix',
           right_on='geolocation_zip_code_prefix', how='left')
)

print(f"Finalny DataFrame: {df.shape}")
print(f"Wierszy: {len(df):,} (powinno ≈ {len(order_items):,} = order_items)")
print(f"\nPolskie tłumaczenia:")
print(f"  Wierszy z polską kategorią:  {df['category_pl'].notna().sum():,} "
      f"({df['category_pl'].notna().mean()*100:.1f}%)")
print(f"  Unikalnych polskich kategorii: {df['category_pl'].nunique()}")

In [ ]:
# ── Podsumowanie łączenia ──
print("=== PODSUMOWANIE ŁĄCZENIA TABEL ===")
print(f"Finalny DataFrame: {df.shape}")
print(f"Kolumny ({df.shape[1]}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nBraki danych (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))


## Etap 1: EDA

**Cele:**
- Jak wygląda sprzedaż w czasie?
- Jakie są rozkłady cen, dostawy, ocen?
- Gdzie są problemy w danych?
- Jakie wzorce można wykorzystać w modelu?


In [ ]:
# ── Podstawowe info ──
print("=== KSZTAŁT DANYCH ===")
print(f"Wierszy: {len(df):,}")
print(f"Kolumn: {df.shape[1]}")


### Data cleaning

In [ ]:
# ── Konwersja dat ──
# errors='coerce' → uszkodzone daty staną się NaT zamiast wyrzucić błąd
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date'
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("Daty skonwertowane ✓")


In [ ]:
# ── Status zamówień ──
status_counts = df['order_status'].value_counts(dropna=False)
status_share = (status_counts / len(df) * 100).round(2)
print("=== STATUS ZAMÓWIEŃ ===")
print(status_share)
print(f"\nUdział delivered: {status_share.get('delivered', 0):.1f}%")


In [ ]:
# ── Filtrowanie do delivered ──
# NIE robimy df.dropna() na wszystkim — tylko filtrujemy po statusie
delivered_df = df[df['order_status'] == 'delivered'].copy()

print(f"Wszystkie rekordy: {len(df):,}")
print(f"Delivered: {len(delivered_df):,}")
print(f"Udział: {len(delivered_df)/len(df)*100:.1f}%")
print(f"\nZakres dat:")
print(f"  Od: {delivered_df['order_purchase_timestamp'].min()}")
print(f"  Do: {delivered_df['order_purchase_timestamp'].max()}")


In [ ]:
# ── Cechy dostawy ──
delivered_df['delivery_days'] = (
    delivered_df['order_delivered_customer_date'] - delivered_df['order_purchase_timestamp']
).dt.days

delivered_df['delivery_vs_estimate_days'] = (
    delivered_df['order_delivered_customer_date'] - delivered_df['order_estimated_delivery_date']
).dt.days

print("Czas dostawy (dni):")
print(delivered_df['delivery_days'].describe().round(1))


In [ ]:
# ── Podstawowe statystyki ──
print("=== PODSUMOWANIE DANYCH ===")
print(f"Unikalne zamówienia: {delivered_df['order_id'].nunique():,}")
print(f"Unikalne produkty: {delivered_df['product_id'].nunique():,}")
print(f"Unikalne kategorie: {delivered_df['category_pl'].nunique()}")
print(f"Unikalni klienci: {delivered_df['customer_unique_id'].nunique():,}")
print(f"Unikalni sprzedawcy: {delivered_df['seller_id'].nunique():,}")
print(f"Łączny przychód: {delivered_df['price'].sum():,.2f} BRL")
print(f"Średnia cena: {delivered_df['price'].mean():.2f} BRL")
print(f"Średni koszt dostawy: {delivered_df['freight_value'].mean():.2f} BRL")
print(f"Średni czas dostawy: {delivered_df['delivery_days'].mean():.1f} dni")
print(f"Średnia ocena: {delivered_df['review_score'].mean():.2f}")


### Wizualizacje EDA

In [ ]:
# ── Rozkład cen ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(delivered_df['price'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Rozkład cen — pełny')
axes[0].set_xlabel('Cena (BRL)')

price_99 = delivered_df['price'].quantile(0.99)
sns.histplot(delivered_df[delivered_df['price'] <= price_99]['price'], bins=50, kde=True, ax=axes[1])
axes[1].set_title(f'Rozkład cen — do 99 percentyla ({price_99:.0f} BRL)')
axes[1].set_xlabel('Cena (BRL)')

sns.boxplot(x=delivered_df['price'], ax=axes[2])
axes[2].set_title('Boxplot cen')

plt.tight_layout()
plt.show()


In [ ]:
# ── Rozkład dostawy ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(delivered_df['delivery_days'].dropna(), bins=40, kde=True, ax=axes[0])
axes[0].set_title('Rozkład czasu dostawy')
axes[0].set_xlabel('Dni')

sns.histplot(delivered_df['delivery_vs_estimate_days'].dropna(), bins=40, kde=True, ax=axes[1])
axes[1].set_title('Dostawa vs estymata (ujemne = szybciej niż obiecano)')
axes[1].set_xlabel('Dni')
axes[1].axvline(0, color='red', linestyle='--', label='Na czas')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Top 15 kategorii ──
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_count = delivered_df['category_pl'].value_counts().head(15).sort_values()
top_count.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 kategorii (PL) — liczba pozycji')
axes[0].set_xlabel('Liczba')

top_revenue = (
    delivered_df.groupby('category_pl')['price']
    .sum().sort_values(ascending=False).head(15).sort_values()
)
top_revenue.plot(kind='barh', ax=axes[1], color='green')
axes[1].set_title('Top 15 kategorii (PL) — przychód (BRL)')
axes[1].set_xlabel('Przychód')

plt.tight_layout()
plt.show()


In [ ]:
# ── Sprzedaż w czasie ──
delivered_df['purchase_month'] = delivered_df['order_purchase_timestamp'].dt.to_period('M')

monthly_data = (
    delivered_df.groupby('purchase_month')
    .agg(
        units_sold=('product_id', 'count'),
        revenue=('price', 'sum'),
        unique_orders=('order_id', 'nunique'),
        avg_review=('review_score', 'mean')
    )
    .reset_index()
)
monthly_data['purchase_month'] = monthly_data['purchase_month'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

sns.lineplot(data=monthly_data, x='purchase_month', y='units_sold', marker='o', ax=axes[0, 0])
axes[0, 0].set_title('Miesięczna sprzedaż (sztuki)')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly_data, x='purchase_month', y='revenue', marker='o', color='green', ax=axes[0, 1])
axes[0, 1].set_title('Miesięczny przychód (BRL)')
axes[0, 1].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly_data, x='purchase_month', y='unique_orders', marker='o', color='orange', ax=axes[1, 0])
axes[1, 0].set_title('Miesięczna liczba unikalnych zamówień')
axes[1, 0].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly_data, x='purchase_month', y='avg_review', marker='o', color='purple', ax=axes[1, 1])
axes[1, 1].set_title('Średnia ocena (review score)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_ylim(1, 5)

plt.tight_layout()
plt.show()


In [ ]:
# ── Top 10 kategorii w czasie ──
top_10_cats = delivered_df['category_pl'].value_counts().head(10).index

monthly_top = (
    delivered_df[delivered_df['category_pl'].isin(top_10_cats)]
    .groupby(['purchase_month', 'category_pl'])
    .size()
    .reset_index(name='units_sold')
)
monthly_top['purchase_month'] = monthly_top['purchase_month'].astype(str)

plt.figure(figsize=(16, 8))
sns.lineplot(data=monthly_top, x='purchase_month', y='units_sold',
             hue='category_pl')
plt.title('Miesięczna sprzedaż — Top 10 kategorii (PL)')
plt.xticks(rotation=45)
plt.legend(title='Kategoria (PL)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# ── Metody płatności i oceny ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

delivered_df['payment_type_main'].value_counts().plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Główna metoda płatności')
axes[0].tick_params(axis='x', rotation=45)

delivered_df['payment_installments_max'].clip(upper=12).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Liczba rat (max 12+)')

delivered_df['review_score'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='gold', edgecolor='black')
axes[2].set_title('Rozkład ocen (review_score)')
axes[2].set_xlabel('Ocena')

plt.tight_layout()
plt.show()


## Etap 2: Feature Engineering

### Strategia:
1. Agregacja do granulacji tygodniowej (kategoria × tydzień)
2. Uzupełnienie brakujących tygodni zerami
3. Cechy lag (opóźnione wartości)
4. Cechy rolling (średnia i std krocząca)
5. Momentum i pct_change
6. Cechy czasowe + święta brazylijskie
7. Target encoding + podział train/val/test

### Dlaczego tygodniowa granulacja?
- Dziennie: za mało obserwacji na kategorię → szum
- Miesięcznie: za grubo → tracimy detale sezonowe
- Tygodniowa: sweet spot dla ~70 kategorii × ~100 tygodni


In [ ]:
# ── Przygotowanie danych ──
ts_df = delivered_df[
    ['order_purchase_timestamp', 'category_pl',
     'order_id', 'product_id', 'price', 'freight_value',
     'review_score', 'customer_state', 'seller_state',
     'payment_value_total', 'payment_installments_max']
].copy()

# Usuwamy braki TYLKO w kategorii
ts_df = ts_df.dropna(subset=['category_pl']).copy()
print(f"Rekordów po usunięciu braków kategorii: {len(ts_df):,}")
print(f"Unikalnych kategorii: {ts_df['category_pl'].nunique()}")


### Krok 1 — Agregacja tygodniowa

In [ ]:
ts_df['purchase_week_start'] = (
    ts_df['order_purchase_timestamp']
    .dt.to_period('W')
    .apply(lambda r: r.start_time)
)

weekly_df = (
    ts_df.groupby(['category_pl', 'purchase_week_start'])
    .agg(
        units_sold=('product_id', 'count'),
        unique_orders=('order_id', 'nunique'),
        revenue=('price', 'sum'),
        avg_price=('price', 'mean'),
        avg_freight=('freight_value', 'mean'),
        avg_review=('review_score', 'mean'),
        avg_payment_value=('payment_value_total', 'mean'),
        avg_installments=('payment_installments_max', 'mean')
    )
    .reset_index()
)

print(f"weekly_df: {weekly_df.shape}")
print(f"Zakres: {weekly_df['purchase_week_start'].min()} → {weekly_df['purchase_week_start'].max()}")
print(f"Kategorii: {weekly_df['category_pl'].nunique()}")


### Krok 2 — Uzupełnienie brakujących tygodni zerami

Jeśli kategoria nie miała sprzedaży w danym tygodniu, ten tydzień nie istnieje w danych.
Model musi wiedzieć, że sprzedaż = 0 żeby lagi i rolling liczyły się poprawnie.


In [ ]:
all_categories = weekly_df['category_pl'].unique()
all_weeks = pd.date_range(
    start=weekly_df['purchase_week_start'].min(),
    end=weekly_df['purchase_week_start'].max(),
    freq='W-MON'
)

full_index = pd.MultiIndex.from_product(
    [all_categories, all_weeks],
    names=['category_pl', 'purchase_week_start']
)

weekly_full = (
    weekly_df.set_index(['category_pl', 'purchase_week_start'])
    .reindex(full_index)
    .reset_index()
)

for col in ['units_sold', 'unique_orders', 'revenue']:
    weekly_full[col] = weekly_full[col].fillna(0)
for col in ['avg_price', 'avg_freight', 'avg_review', 'avg_payment_value', 'avg_installments']:
    weekly_full[col] = weekly_full[col].fillna(0)

weekly_full = weekly_full.sort_values(
    ['category_pl', 'purchase_week_start']
).reset_index(drop=True)

print(f"Po uzupełnieniu: {weekly_full.shape}")
print(f"Tygodnie z units_sold=0: {(weekly_full['units_sold']==0).sum()} "
      f"({(weekly_full['units_sold']==0).mean()*100:.1f}%)")


### Krok 3 — Cechy lag

Lag = wartość z N tygodni temu. Sprzedaż silnie koreluje z historią.
`shift(N)` zapobiega data leakage.


In [ ]:
for lag in [1, 2, 4, 8]:
    weekly_full[f'lag_{lag}'] = (
        weekly_full.groupby('category_pl')['units_sold'].shift(lag)
    )

weekly_full['revenue_lag_1'] = weekly_full.groupby('category_pl')['revenue'].shift(1)
weekly_full['orders_lag_1'] = weekly_full.groupby('category_pl')['unique_orders'].shift(1)
weekly_full['avg_price_lag_1'] = weekly_full.groupby('category_pl')['avg_price'].shift(1)

lag_cols = [c for c in weekly_full.columns if 'lag' in c]
print(f"Cechy lag ({len(lag_cols)}): {lag_cols}")


### Krok 4 — Cechy rolling

Rolling mean = średnia z ostatnich N tygodni → wygładza szum, pokazuje trend.
`shift(1)` przed rolling → żeby model nie podglądał bieżącego tygodnia.


In [ ]:
for window in [4, 8]:
    weekly_full[f'rolling_mean_{window}'] = (
        weekly_full.groupby('category_pl')['units_sold']
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=2).mean())
    )
    weekly_full[f'rolling_std_{window}'] = (
        weekly_full.groupby('category_pl')['units_sold']
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=2).std())
    )

weekly_full['revenue_roll_mean_4'] = (
    weekly_full.groupby('category_pl')['revenue']
    .transform(lambda x: x.shift(1).rolling(window=4, min_periods=2).mean())
)
weekly_full['revenue_roll_std_4'] = (
    weekly_full.groupby('category_pl')['revenue']
    .transform(lambda x: x.shift(1).rolling(window=4, min_periods=2).std())
)

roll_cols = [c for c in weekly_full.columns if 'roll' in c]
print(f"Cechy rolling ({len(roll_cols)}): {roll_cols}")


### Krok 5 — Momentum i pct_change

- **Momentum** = lag_1 − rolling_mean → powyżej/poniżej trendu?
- **pct_change** = procentowa zmiana tydzień do tygodnia


In [ ]:
weekly_full['momentum_1_4'] = weekly_full['lag_1'] - weekly_full['rolling_mean_4']
weekly_full['momentum_1_8'] = weekly_full['lag_1'] - weekly_full['rolling_mean_8']

weekly_full['units_pct_change_1'] = (
    weekly_full.groupby('category_pl')['units_sold']
    .transform(lambda x: x.shift(1).pct_change())
)
weekly_full['revenue_pct_change_1'] = (
    weekly_full.groupby('category_pl')['revenue']
    .transform(lambda x: x.shift(1).pct_change())
)

print("Momentum i pct_change dodane ✓")


### Krok 6 — Cechy czasowe + święta brazylijskie

Dane są z Brazylii, więc dodajemy brazylijskie święta. Święta mocno wpływają na sprzedaż 
(Black Friday, Dzień Matki, Boże Narodzenie).


In [ ]:
weekly_full['year'] = weekly_full['purchase_week_start'].dt.year
weekly_full['month'] = weekly_full['purchase_week_start'].dt.month
weekly_full['quarter'] = weekly_full['purchase_week_start'].dt.quarter
weekly_full['week_of_year'] = weekly_full['purchase_week_start'].dt.isocalendar().week.astype(int)
weekly_full['is_month_start'] = weekly_full['purchase_week_start'].dt.is_month_start.astype(int)
weekly_full['is_month_end'] = weekly_full['purchase_week_start'].dt.is_month_end.astype(int)

br_holidays = [
    '2016-12-25', '2017-01-01', '2017-12-25', '2018-01-01', '2018-12-25',
    '2016-05-08', '2017-05-14', '2018-05-13',      # Dzień Matki
    '2017-02-27', '2017-02-28', '2018-02-12', '2018-02-13',  # Karnawał
    '2016-11-25', '2017-11-24', '2018-11-23',      # Black Friday
    '2016-10-12', '2017-10-12', '2018-10-12',      # Dzień Dziecka
]
br_holidays_dt = pd.to_datetime(br_holidays)

def is_holiday_week(week_start):
    week_dates = pd.date_range(week_start, periods=7)
    return int(any(h in week_dates for h in br_holidays_dt))

weekly_full['is_holiday'] = weekly_full['purchase_week_start'].apply(is_holiday_week)

print(f"Cechy czasowe dodane ✓")
print(f"Tygodnie ze świętem: {weekly_full['is_holiday'].sum()}")


### Krok 7 — Czyszczenie, encoding, podział danych

In [ ]:
# ── Czyszczenie inf ──
# pct_change() tworzy inf gdy dzielimy przez 0 — zamieniamy na NaN
weekly_full = weekly_full.replace([np.inf, -np.inf], np.nan)

# Usuwamy wiersze bez kluczowych lagów (pierwsze tygodnie każdej kategorii)
feature_df = weekly_full.dropna(
    subset=['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_mean_8']
).copy()

print(f"weekly_full: {weekly_full.shape}")
print(f"feature_df: {feature_df.shape}")
print(f"Usunięto: {len(weekly_full) - len(feature_df)} wierszy (brak lagów)")


In [ ]:
from sklearn.preprocessing import LabelEncoder

# ── Label encoding kategorii ──
le = LabelEncoder()
feature_df['category_enc'] = le.fit_transform(feature_df['category_pl'])

# ── Podział chronologiczny 70/15/15 ──
sorted_dates = sorted(feature_df['purchase_week_start'].unique())
n_dates = len(sorted_dates)

train_end_date = sorted_dates[int(n_dates * 0.70) - 1]
valid_end_date = sorted_dates[int(n_dates * 0.85) - 1]

train_mask = feature_df['purchase_week_start'] <= train_end_date
val_mask = (feature_df['purchase_week_start'] > train_end_date) & (feature_df['purchase_week_start'] <= valid_end_date)
test_mask = feature_df['purchase_week_start'] > valid_end_date

print(f"Train: {train_mask.sum():>6,} wierszy  ({feature_df.loc[train_mask, 'purchase_week_start'].min().date()} → {feature_df.loc[train_mask, 'purchase_week_start'].max().date()})")
print(f"Val:   {val_mask.sum():>6,} wierszy  ({feature_df.loc[val_mask, 'purchase_week_start'].min().date()} → {feature_df.loc[val_mask, 'purchase_week_start'].max().date()})")
print(f"Test:  {test_mask.sum():>6,} wierszy  ({feature_df.loc[test_mask, 'purchase_week_start'].min().date()} → {feature_df.loc[test_mask, 'purchase_week_start'].max().date()})")


In [ ]:
# ── Target encoding (TYLKO z train!) ──
# Kodujemy kategorię jako jej średnią sprzedaż z train setu.
# Z val/test = data leakage!

if 'category_mean_units' in feature_df.columns:
    feature_df = feature_df.drop(columns=['category_mean_units'])

target_enc_map = (
    feature_df[feature_df['purchase_week_start'] <= train_end_date]
    .groupby('category_pl')['units_sold']
    .mean()
    .reset_index()
    .rename(columns={'units_sold': 'category_mean_units'})
)

feature_df = feature_df.merge(target_enc_map, on='category_pl', how='left')

global_mean = feature_df[feature_df['purchase_week_start'] <= train_end_date]['units_sold'].mean()
feature_df['category_mean_units'] = feature_df['category_mean_units'].fillna(global_mean)

# Przelicz maski (merge zmienia indeks)
train_mask = feature_df['purchase_week_start'] <= train_end_date
val_mask = (feature_df['purchase_week_start'] > train_end_date) & (feature_df['purchase_week_start'] <= valid_end_date)
test_mask = feature_df['purchase_week_start'] > valid_end_date

print(f"Target encoding — top 5 kategorii:")
print(target_enc_map.sort_values('category_mean_units', ascending=False).head())


In [ ]:
# ── Definicja cech ──
FEATURES = [
    # Lagi units_sold
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    # Lagi dodatkowe
    'revenue_lag_1', 'orders_lag_1', 'avg_price_lag_1',
    # Rolling units_sold
    'rolling_mean_4', 'rolling_std_4', 'rolling_mean_8', 'rolling_std_8',
    # Rolling revenue
    'revenue_roll_mean_4', 'revenue_roll_std_4',
    # Momentum
    'momentum_1_4', 'momentum_1_8',
    # Pct change
    'units_pct_change_1', 'revenue_pct_change_1',
    # Bazowe
    'avg_price', 'avg_freight',
    # Czas
    'year', 'month', 'quarter', 'week_of_year',
    'is_month_start', 'is_month_end', 'is_holiday',
    # Kategoria
    'category_enc', 'category_mean_units',
]

TARGET = 'units_sold'

# Wypełnij NaN w niestabilnych cechach
for col in ['units_pct_change_1', 'revenue_pct_change_1',
            'rolling_std_4', 'rolling_std_8', 'revenue_roll_std_4']:
    feature_df[col] = feature_df[col].fillna(0)

# ── Finalne X i y ──
X_train = feature_df.loc[train_mask, FEATURES].copy()
y_train = feature_df.loc[train_mask, TARGET].copy()
X_val = feature_df.loc[val_mask, FEATURES].copy()
y_val = feature_df.loc[val_mask, TARGET].copy()
X_test = feature_df.loc[test_mask, FEATURES].copy()
y_test = feature_df.loc[test_mask, TARGET].copy()

print(f"Liczba cech: {len(FEATURES)}")
print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}  y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  y_test:  {y_test.shape}")

for name, X in [('train', X_train), ('val', X_val), ('test', X_test)]:
    print(f"NaN w X_{name}: {X.isnull().sum().sum()}")


## Analiza korelacji

**Po co?**
1. Które cechy najsilniej wpływają na target?
2. Które cechy duplikują informację (multicolinearność)?

Dla regresji liniowej multicolinearność = problem (niestabilne współczynniki).
Dla modeli drzewiastych (LightGBM, XGBoost, CatBoost) = mniejszy problem.


In [ ]:
# ── Korelacja z targetem ──
corr_data = feature_df[FEATURES + [TARGET]].copy()
target_corr = corr_data.corr()[TARGET].drop(TARGET).sort_values(ascending=False)

print("=== KORELACJA CECH Z TARGETEM ===")
print("\nNajsilniej skorelowane:")
print(target_corr.head(10).round(3).to_string())
print("\nNajsłabiej skorelowane:")
print(target_corr.tail(5).round(3).to_string())


In [ ]:
# ── Wykres korelacji z targetem ──
plt.figure(figsize=(12, 10))
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in target_corr.values]
target_corr.plot(kind='barh', color=colors)
plt.title(f'Korelacja cech z targetem ({TARGET})')
plt.xlabel('Korelacja Pearsona')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# ── Heatmapa korelacji ──
plt.figure(figsize=(20, 16))
corr_matrix = corr_data[FEATURES].corr()

sns.heatmap(
    corr_matrix, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 7}
)
plt.title('Macierz korelacji między cechami')
plt.tight_layout()
plt.show()

# Pary o wysokiej korelacji
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

if high_corr:
    print("\n⚠ Pary cech o korelacji > 0.9:")
    for c1, c2, corr in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {c1} ↔ {c2}: {corr:.3f}")


## Etap 3: Modele

Porównujemy 8 modeli:
1. **Baseline (lag_1)** — predykcja = sprzedaż z poprzedniego tygodnia
2. **Ridge Regression** — regresja liniowa z regularyzacją L2
3. **Random Forest** — las drzew decyzyjnych
4. **LightGBM** — gradient boosting (Microsoft)
5. **XGBoost** — gradient boosting (rozwijany wcześniej)
6. **CatBoost** — gradient boosting (Yandex)
7. **LSTM** — sieć neuronowa rekurencyjna (Keras/TensorFlow)
8. **Prophet** — model szeregu czasowego (Facebook/Meta)

### Metryki:
- **RMSE** — kara kwadratowa za duże błędy
- **MAE** — średni błąd bezwzględny
- **MAPE** — błąd procentowy (ignoruje zera)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(name, y_true, y_pred, dataset=''):
    r = rmse(y_true, y_pred)
    m = mean_absolute_error(y_true, y_pred)
    mp = mape(y_true, y_pred)
    print(f"{name:25s} [{dataset:10s}] RMSE: {r:8.3f} | MAE: {m:8.3f} | MAPE: {mp:6.1f}%")
    return {'model': name, 'dataset': dataset, 'RMSE': r, 'MAE': m, 'MAPE': mp}

# Wypełnij NaN zerami (niektóre modele nie akceptują braków)
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)
X_test = X_test.fillna(0)

results = []
print("Funkcje ewaluacji gotowe ✓")


### Model 1: Baseline (lag_1)

Najprostszy "model" — predykcja = sprzedaż z poprzedniego tygodnia.
Każdy poważny model musi bić baseline.


In [ ]:
results.append(evaluate('Baseline (lag_1)', y_val, X_val['lag_1'], 'Validation'))
results.append(evaluate('Baseline (lag_1)', y_test, X_test['lag_1'], 'Test'))


### Model 2: Ridge Regression

Regresja liniowa z regularyzacją L2. Prosty, interpretowalny model.
Pokazuje że liniowe zależności nie wystarczą dla tego problemu.


In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

ridge_pred_val = np.maximum(ridge.predict(X_val), 0)
ridge_pred_test = np.maximum(ridge.predict(X_test), 0)

results.append(evaluate('Ridge', y_val, ridge_pred_val, 'Validation'))
results.append(evaluate('Ridge', y_test, ridge_pred_test, 'Test'))


### Model 3: Random Forest

Las drzew decyzyjnych — bierze losowe próbki danych i cech, buduje wiele drzew,
a predykcja = średnia z wszystkich. Odporny na overfitting.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred_val = np.maximum(rf_model.predict(X_val), 0)
rf_pred_test = np.maximum(rf_model.predict(X_test), 0)

results.append(evaluate('Random Forest', y_val, rf_pred_val, 'Validation'))
results.append(evaluate('Random Forest', y_test, rf_pred_test, 'Test'))


### Model 4: LightGBM

Gradient boosting Microsoftu — szybki, wydajny, często wygrywa na danych tabelarycznych.
Early stopping zatrzymuje trening gdy wynik na val przestaje się poprawiać.


In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0)
    ]
)

lgb_pred_val = np.maximum(lgb_model.predict(X_val), 0)
lgb_pred_test = np.maximum(lgb_model.predict(X_test), 0)

results.append(evaluate('LightGBM', y_val, lgb_pred_val, 'Validation'))
results.append(evaluate('LightGBM', y_test, lgb_pred_test, 'Test'))


### Model 5: XGBoost

Gradient boosting — klasyk, bardzo wydajny na danych tabelarycznych.


In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_pred_val = np.maximum(xgb_model.predict(X_val), 0)
xgb_pred_test = np.maximum(xgb_model.predict(X_test), 0)

results.append(evaluate('XGBoost', y_val, xgb_pred_val, 'Validation'))
results.append(evaluate('XGBoost', y_test, xgb_pred_test, 'Test'))


### Model 6: CatBoost

Gradient boosting Yandeksu — dobrze radzi sobie z cechami kategorycznymi,
wymaga mniej strojenia niż XGBoost/LightGBM.


In [ ]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=0
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
    early_stopping_rounds=50
)

cat_pred_val = np.maximum(cat_model.predict(X_val), 0)
cat_pred_test = np.maximum(cat_model.predict(X_test), 0)

results.append(evaluate('CatBoost', y_val, cat_pred_val, 'Validation'))
results.append(evaluate('CatBoost', y_test, cat_pred_test, 'Test'))


### Feature importance — modele drzewiaste

Porównujemy które cechy są najważniejsze dla każdego modelu.


In [ ]:
# ── Feature importance: porównanie 4 modeli ──
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

importance_models = [
    ('Random Forest', rf_model.feature_importances_, axes[0, 0]),
    ('LightGBM', lgb_model.feature_importances_, axes[0, 1]),
    ('XGBoost', xgb_model.feature_importances_, axes[1, 0]),
    ('CatBoost', cat_model.get_feature_importance(), axes[1, 1]),
]

for name, importances, ax in importance_models:
    imp_df = pd.DataFrame({
        'feature': FEATURES, 'importance': importances
    }).sort_values('importance', ascending=False).head(15)
    sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis', ax=ax)
    ax.set_title(f'{name} — Top 15 cech')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()


### Model 7: LSTM (sieć neuronowa rekurencyjna)

**LSTM** (Long Short-Term Memory) to sieć neuronowa zaprojektowana do danych sekwencyjnych — 
idealna dla szeregów czasowych. Różni się od reszty modeli tym, że uczy się "pamiętać" 
wzorce w sekwencji.

**Przygotowanie danych dla LSTM:**
- Normalizacja cech do [0,1] (MinMaxScaler) — sieci neuronowe tego wymagają
- Reshape do 3D: (samples, timesteps, features) — LSTM przetwarza sekwencje
- Używamy 1 timestep (1 tydzień) bo nasze cechy już zawierają lagi i rolling

**Uwaga:** LSTM jest zwykle GORSZY od XGBoost/LightGBM na tak małych datasetach tabelarycznych.
Sprawdzamy go dla porównania — na większych danych (miliony wierszy) byłby mocniejszy.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ── Przygotowanie danych dla LSTM ──
# 1) Normalizacja (MinMaxScaler — skaluje do [0,1])
scaler_lstm = MinMaxScaler()
X_train_s = scaler_lstm.fit_transform(X_train)
X_val_s = scaler_lstm.transform(X_val)
X_test_s = scaler_lstm.transform(X_test)

# 2) Reshape do 3D: (samples, timesteps=1, features)
X_tr_lstm = X_train_s.reshape(X_train_s.shape[0], 1, X_train_s.shape[1])
X_va_lstm = X_val_s.reshape(X_val_s.shape[0], 1, X_val_s.shape[1])
X_te_lstm = X_test_s.reshape(X_test_s.shape[0], 1, X_test_s.shape[1])

print(f"X_tr_lstm shape: {X_tr_lstm.shape}")
print(f"X_va_lstm shape: {X_va_lstm.shape}")
print(f"X_te_lstm shape: {X_te_lstm.shape}")


In [ ]:
# ── Budowa modelu LSTM ──
tf.random.set_seed(42)

model_lstm = Sequential([
    LSTM(64, input_shape=(1, X_tr_lstm.shape[2]), return_sequences=False),
    Dropout(0.2),       # regularyzacja — losowo wyłącza 20% neuronów
    Dense(32, activation='relu'),
    Dense(1)            # 1 neuron wyjściowy — predykcja sprzedaży
])

model_lstm.compile(optimizer='adam', loss='mse')

# Early stopping — zatrzymaj gdy val_loss przestaje spadać
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model_lstm.fit(
    X_tr_lstm, y_train.values,
    validation_data=(X_va_lstm, y_val.values),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

print(f"Trenowanie zakończone po {len(history.history['loss'])} epokach")


In [ ]:
# ── Predykcje i ewaluacja LSTM ──
lstm_pred_val = np.maximum(model_lstm.predict(X_va_lstm, verbose=0).flatten(), 0)
lstm_pred_test = np.maximum(model_lstm.predict(X_te_lstm, verbose=0).flatten(), 0)

results.append(evaluate('LSTM', y_val, lstm_pred_val, 'Validation'))
results.append(evaluate('LSTM', y_test, lstm_pred_test, 'Test'))


In [ ]:
# ── Krzywa uczenia LSTM ──
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('LSTM — krzywa uczenia')
plt.xlabel('Epoka')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Model 8: Prophet (model szeregu czasowego)

Prophet działa inaczej niż reszta — trenuje osobny model per kategoria i prognozuje przyszłość
na podstawie trendu + sezonowości. Nie używa lagów ani innych cech — tylko datę i sprzedaż.

Dla porównania trenujemy Propheta na TOP 10 kategorii.


In [ ]:
from prophet import Prophet
import logging

# Wyciszamy logi Propheta
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

# ── Prophet per kategoria (top 10) ──
top_categories = (
    feature_df.loc[train_mask]
    .groupby('category_pl')['units_sold']
    .sum().nlargest(10).index.tolist()
)

print(f"Trenujemy Propheta dla {len(top_categories)} top kategorii...")

prophet_preds_val = []
prophet_preds_test = []
prophet_true_val = []
prophet_true_test = []

for cat in top_categories:
    # Dane train + val + test dla tej kategorii
    cat_data = feature_df[feature_df['category_pl'] == cat][
        ['purchase_week_start', 'units_sold']
    ].rename(columns={'purchase_week_start': 'ds', 'units_sold': 'y'}).copy()
    
    cat_train = cat_data[cat_data['ds'] <= train_end_date]
    cat_val = cat_data[(cat_data['ds'] > train_end_date) & (cat_data['ds'] <= valid_end_date)]
    cat_test = cat_data[cat_data['ds'] > valid_end_date]
    
    if len(cat_train) < 20:
        continue
    
    m = Prophet(weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=True)
    m.fit(cat_train)
    
    # Predykcja na val
    if len(cat_val) > 0:
        forecast_val = m.predict(cat_val[['ds']])
        prophet_preds_val.extend(np.maximum(forecast_val['yhat'].values, 0))
        prophet_true_val.extend(cat_val['y'].values)
    
    # Predykcja na test
    if len(cat_test) > 0:
        forecast_test = m.predict(cat_test[['ds']])
        prophet_preds_test.extend(np.maximum(forecast_test['yhat'].values, 0))
        prophet_true_test.extend(cat_test['y'].values)

print(f"\nProphet — wyniki (top 10 kategorii):")
results.append(evaluate('Prophet (top 10)', prophet_true_val, prophet_preds_val, 'Validation'))
results.append(evaluate('Prophet (top 10)', prophet_true_test, prophet_preds_test, 'Test'))

print("\n⚠ Uwaga: Prophet ewaluowany tylko na top 10 kategorii (każda wymaga osobnego modelu).")
print("   Wyniki nie są w pełni porównywalne z modelami ML które widzą wszystkie kategorie.")


## Tuning hiperparametrów (Optuna)

Strojąc 3 najlepsze modele (LightGBM, XGBoost, CatBoost) za pomocą biblioteki **Optuna**.

### Co to jest Optuna?
Biblioteka do automatycznego doboru hiperparametrów. Zamiast ręcznie próbować "może learning_rate=0.05? a może 0.1?", 
Optuna sama testuje różne kombinacje używając inteligentnego algorytmu (TPE — Tree-structured Parzen Estimator).

### Jak działa?
1. Definiujesz przestrzeń poszukiwań (np. `learning_rate` z zakresu 0.01-0.3)
2. Definiujesz funkcję celu (objective) — co minimalizujemy (RMSE)
3. Optuna uruchamia N prób (trials), za każdym razem testując inne hiperparametry
4. Po każdej próbie uczy się które kombinacje działają lepiej

### Dlaczego te 3 modele?
LightGBM, XGBoost i CatBoost to gradient boosting — najlepsze dla danych tabelarycznych.
Mają wiele hiperparametrów do strojenia, więc Optuna przyniesie największy zysk.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # mniej outputu

# Liczba prób per model — zwiększ do 50-100 dla lepszych wyników
N_TRIALS = 30

print(f"Tuning {N_TRIALS} prób per model")
print("Czas wykonania: 2-5 minut per model")


### Tuning LightGBM

In [ ]:
def objective_lgb(trial):
    """Funkcja celu dla LightGBM — Optuna ją minimalizuje."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1
    }
    
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    pred = np.maximum(model.predict(X_val), 0)
    return rmse(y_val, pred)

study_lgb = optuna.create_study(direction='minimize', study_name='lightgbm')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nNajlepsze RMSE: {study_lgb.best_value:.4f}")
print(f"Najlepsze parametry:")
for k, v in study_lgb.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
# ── Trenowanie LightGBM z najlepszymi parametrami ──
best_params_lgb = study_lgb.best_params.copy()
best_params_lgb.update({'random_state': 42, 'n_jobs': -1, 'verbosity': -1})

lgb_tuned = lgb.LGBMRegressor(**best_params_lgb)
lgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)

lgb_tuned_pred_val = np.maximum(lgb_tuned.predict(X_val), 0)
lgb_tuned_pred_test = np.maximum(lgb_tuned.predict(X_test), 0)

results.append(evaluate('LightGBM (tuned)', y_val, lgb_tuned_pred_val, 'Validation'))
results.append(evaluate('LightGBM (tuned)', y_test, lgb_tuned_pred_test, 'Test'))


### Tuning XGBoost

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'objective': 'reg:squarederror',
        'random_state': 42,
        'n_jobs': -1,
        'early_stopping_rounds': 30,
        'verbosity': 0
    }
    
    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = np.maximum(model.predict(X_val), 0)
    return rmse(y_val, pred)

study_xgb = optuna.create_study(direction='minimize', study_name='xgboost')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nNajlepsze RMSE: {study_xgb.best_value:.4f}")
print(f"Najlepsze parametry:")
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
# ── Trenowanie XGBoost z najlepszymi parametrami ──
best_params_xgb = study_xgb.best_params.copy()
best_params_xgb.update({
    'objective': 'reg:squarederror',
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 30,
    'verbosity': 0
})

xgb_tuned = XGBRegressor(**best_params_xgb)
xgb_tuned.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_tuned_pred_val = np.maximum(xgb_tuned.predict(X_val), 0)
xgb_tuned_pred_test = np.maximum(xgb_tuned.predict(X_test), 0)

results.append(evaluate('XGBoost (tuned)', y_val, xgb_tuned_pred_val, 'Validation'))
results.append(evaluate('XGBoost (tuned)', y_test, xgb_tuned_pred_test, 'Test'))


### Tuning CatBoost

In [ ]:
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_seed': 42,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'verbose': 0
    }
    
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), 
              use_best_model=True, early_stopping_rounds=30)
    pred = np.maximum(model.predict(X_val), 0)
    return rmse(y_val, pred)

study_cat = optuna.create_study(direction='minimize', study_name='catboost')
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nNajlepsze RMSE: {study_cat.best_value:.4f}")
print(f"Najlepsze parametry:")
for k, v in study_cat.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
# ── Trenowanie CatBoost z najlepszymi parametrami ──
best_params_cat = study_cat.best_params.copy()
best_params_cat.update({
    'random_seed': 42,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'verbose': 0
})

cat_tuned = CatBoostRegressor(**best_params_cat)
cat_tuned.fit(X_train, y_train, eval_set=(X_val, y_val),
              use_best_model=True, early_stopping_rounds=30)

cat_tuned_pred_val = np.maximum(cat_tuned.predict(X_val), 0)
cat_tuned_pred_test = np.maximum(cat_tuned.predict(X_test), 0)

results.append(evaluate('CatBoost (tuned)', y_val, cat_tuned_pred_val, 'Validation'))
results.append(evaluate('CatBoost (tuned)', y_test, cat_tuned_pred_test, 'Test'))


### Analiza wyników Optuna

Wizualizacja procesu strojenia — co Optuna nauczyła się o przestrzeni hiperparametrów.


In [ ]:
# ── Historia optymalizacji ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (study, name) in zip(axes, [
    (study_lgb, 'LightGBM'),
    (study_xgb, 'XGBoost'),
    (study_cat, 'CatBoost')
]):
    trials = study.trials
    values = [t.value for t in trials if t.value is not None]
    best_so_far = [min(values[:i+1]) for i in range(len(values))]
    
    ax.plot(values, 'o-', alpha=0.4, label='Każda próba')
    ax.plot(best_so_far, 'r-', linewidth=2, label='Najlepszy wynik')
    ax.set_title(f'{name} — historia optymalizacji')
    ax.set_xlabel('Numer próby')
    ax.set_ylabel('RMSE (Validation)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Ważność hiperparametrów (Optuna) ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (study, name) in zip(axes, [
    (study_lgb, 'LightGBM'),
    (study_xgb, 'XGBoost'),
    (study_cat, 'CatBoost')
]):
    try:
        importances = optuna.importance.get_param_importances(study)
        params = list(importances.keys())
        values = list(importances.values())
        
        ax.barh(params, values, color='steelblue')
        ax.set_title(f'{name} — ważność hiperparametrów')
        ax.set_xlabel('Importance')
    except Exception as e:
        ax.text(0.5, 0.5, f'Błąd: {e}', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()


In [ ]:
# ── Porównanie: przed vs po tuningu ──
# Budujemy results_df_temp lokalnie z listy results (która istnieje w pamięci)
results_df_temp = pd.DataFrame(results)

def get_rmse(model_name):
    """Pobiera RMSE z Test setu dla danego modelu."""
    rows = results_df_temp[
        (results_df_temp['model'] == model_name) & 
        (results_df_temp['dataset'] == 'Test')
    ]
    return rows['RMSE'].values[0] if len(rows) > 0 else np.nan

comparison = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'RMSE przed': [
        get_rmse('LightGBM'),
        get_rmse('XGBoost'),
        get_rmse('CatBoost'),
    ],
    'RMSE po': [
        rmse(y_test, lgb_tuned_pred_test),
        rmse(y_test, xgb_tuned_pred_test),
        rmse(y_test, cat_tuned_pred_test),
    ]
})
comparison['Poprawa %'] = ((comparison['RMSE przed'] - comparison['RMSE po']) / comparison['RMSE przed'] * 100).round(2)

print("=== TUNING — PRZED vs PO ===")
print(comparison.to_string(index=False))

# Wizualizacja
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width/2, comparison['RMSE przed'], width, label='Przed tuningiem', color='#e74c3c')
ax.bar(x + width/2, comparison['RMSE po'], width, label='Po tuningu', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'])
ax.set_ylabel('RMSE (Test)')
ax.set_title('Wpływ tuningu hiperparametrów (Optuna)')
ax.legend()

for i_bar, (przed, po) in enumerate(zip(comparison['RMSE przed'], comparison['RMSE po'])):
    if not np.isnan(przed):
        ax.text(i_bar - width/2, przed, f'{przed:.2f}', ha='center', va='bottom', fontsize=9)
    ax.text(i_bar + width/2, po, f'{po:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


## Porównanie modeli

In [ ]:
# ── Tabela wyników ──
results_df = pd.DataFrame(results)
print("=" * 80)
print("PORÓWNANIE MODELI")
print("=" * 80)
print(results_df.to_string(index=False))


In [ ]:
# ── Wizualizacja porównania ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
test_res = results_df[results_df['dataset'] == 'Test'].copy()
colors = sns.color_palette('viridis', n_colors=len(test_res))

for i, metric in enumerate(['RMSE', 'MAE', 'MAPE']):
    axes[i].bar(test_res['model'], test_res[metric], color=colors)
    axes[i].set_title(f'{metric} (Test)')
    axes[i].tick_params(axis='x', rotation=30)
    for j, v in enumerate(test_res[metric].values):
        axes[i].text(j, v + v*0.02, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('Porównanie modeli na zbiorze testowym', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Predykcja vs Rzeczywistość (modele ML) ──
fig, axes = plt.subplots(3, 4, figsize=(22, 14))

models_preds = [
    ('Baseline', np.array(X_test['lag_1'])),
    ('Ridge', ridge_pred_test),
    ('Random Forest', rf_pred_test),
    ('LightGBM', lgb_pred_test),
    ('XGBoost', xgb_pred_test),
    ('CatBoost', cat_pred_test),
    ('LSTM', lstm_pred_test),
    ('LightGBM (tuned)', lgb_tuned_pred_test),
    ('XGBoost (tuned)', xgb_tuned_pred_test),
    ('CatBoost (tuned)', cat_tuned_pred_test),
]

for i, (name, pred) in enumerate(models_preds):
    ax = axes[i // 4, i % 4]
    ax.scatter(y_test, pred, alpha=0.3, s=10)
    max_val = max(y_test.max(), np.max(pred))
    ax.plot([0, max_val], [0, max_val], 'r--', label='Idealna predykcja')
    ax.set_xlabel('Rzeczywiste')
    ax.set_ylabel('Predykcja')
    ax.set_title(name)
    ax.legend()

plt.suptitle('Predykcja vs Rzeczywistość (Test)', fontsize=14, fontweight='bold')
# Ukryj puste subploty
for i in range(len(models_preds), 12):
    axes[i // 4, i % 4].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ── Rozkład reszt ──
n_models = len(models_preds)
n_cols = 4
n_rows = (n_models + n_cols - 1) // n_cols  # zaokrąglenie w górę

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 4 * n_rows))
axes = np.array(axes).reshape(n_rows, n_cols)  # zabezpieczenie dla 1 wiersza

for i, (name, pred) in enumerate(models_preds):
    ax = axes[i // n_cols, i % n_cols]
    residuals = np.array(y_test) - np.array(pred)
    sns.histplot(residuals, bins=50, kde=True, ax=ax)
    ax.axvline(0, color='red', linestyle='--')
    ax.set_title(f'{name} — rozkład reszt')
    ax.set_xlabel('Residual')
    ax.text(0.05, 0.95, f'μ={residuals.mean():.2f}\nσ={residuals.std():.2f}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Ukryj puste subploty
for i in range(n_models, n_rows * n_cols):
    axes[i // n_cols, i % n_cols].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print(f"Liczba modeli: {len(models_preds)}")
for name, _ in models_preds:
    print(f"  - {name}")

## Podsumowanie

### Wnioski z Etapu 3:


In [ ]:
print("=" * 70)
print("PODSUMOWANIE ETAPU 3 — PORÓWNANIE MODELI")
print("=" * 70)

test_res_indexed = results_df[results_df['dataset'] == 'Test'].set_index('model')

print(f"\nNajlepszy model wg RMSE: {test_res_indexed['RMSE'].idxmin()} ({test_res_indexed['RMSE'].min():.3f})")
print(f"Najlepszy model wg MAE:  {test_res_indexed['MAE'].idxmin()} ({test_res_indexed['MAE'].min():.3f})")
print(f"Najlepszy model wg MAPE: {test_res_indexed['MAPE'].idxmin()} ({test_res_indexed['MAPE'].min():.1f}%)")

print(f"\nLiczba cech: {len(FEATURES)}")
print(f"Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")

# Ranking modeli wg RMSE
print(f"\n{'─' * 50}")
print("Ranking modeli wg RMSE (Test):")
print(f"{'─' * 50}")
for rank, (model, row) in enumerate(test_res_indexed.sort_values('RMSE').iterrows(), 1):
    print(f"  {rank}. {model:25s} RMSE: {row['RMSE']:7.3f} | MAE: {row['MAE']:6.3f}")

print("\n" + "=" * 70)
print("GOTOWY DO ETAPU 4 — trendy i stany magazynowe")
print("=" * 70)


## Etap 4: Trendy i stany magazynowe

W tym etapie wykorzystujemy najlepszy model z Etapu 3 (**XGBoost tuned**) do:

1. **Prognozy 4 tygodni do przodu** — predykcja sprzedaży na nadchodzący miesiąc (rekursywnie)
2. **Wykrywania trendów** — porównanie 4tyg vs 4tyg + SARIMA dla top kategorii
3. **Wyznaczenia stanów magazynowych** — hybrydowy Safety Stock oparty na **RMSE modelu** + **wariancji czasu dostawy**
4. **Pipeline CSV** — klient wgrywa magazyn → system zwraca rekomendacje
5. **Eksportu wyników do CSV** — gotowy raport dla działu logistyki

### Kluczowe pojęcia:

**Lead Time** — czas od zamówienia towaru u dostawcy do dostarczenia. Liczymy z prawdziwych danych Olist.

**Safety Stock (hybrydowy)** = `Z × √(LT_avg × RMSE² + D_avg² × LT_std²)`
- `RMSE` = błąd modelu per kategoria
- `LT_avg`, `LT_std` = średnia i odchylenie czasu dostawy
- `D_avg` = średnia prognoza tygodniowa
- **Założenie:** ryzyko = błąd predykcji modelu + niepewność dostawy

Klasyczna metoda zakłada że ryzyko = historyczne wahania popytu (`σ × √LT`). Ale my mamy model, który już wyłapał regularne wzorce — zostaje tylko ryzyko residualne (RMSE) plus niepewność dostawy. To jest **profesjonalne podejście** używane w realnej logistyce.

**Reorder Point** = `(D_avg × LT_avg) + Safety_Stock`
- Poziom zapasu, przy którym należy złożyć kolejne zamówienie

**Z = 1.65** — wartość krytyczna z rozkładu normalnego dla 95% service level


### Krok 1: Predykcja rekursywna 4 tygodni do przodu

**Strategia rekursywna:** model predykuje tydzień 1, ta predykcja staje się `lag_1` dla tygodnia 2,
i tak dalej aż do tygodnia 4.

**Plus:** używamy jednego modelu (XGBoost tuned)  
**Minus:** błędy się kumulują — predykcja tygodnia 4 jest mniej dokładna niż tygodnia 1


In [ ]:
# ── Wybór najlepszego modelu ──
# Z rankingu Etapu 3 — XGBoost (tuned) ma najlepsze RMSE
best_model = xgb_tuned
best_model_name = 'XGBoost (tuned)'

# Liczba tygodni do prognozowania
FORECAST_WEEKS = 4
LEAD_TIME_WEEKS = 1  # Zakładamy 1 tydzień na uzupełnienie zapasów

# Ostatnia data w danych
last_date = feature_df['purchase_week_start'].max()
forecast_start = last_date + pd.Timedelta(weeks=1)

print(f"Model: {best_model_name}")
print(f"Ostatnia data w danych: {last_date.date()}")
print(f"Prognoza od: {forecast_start.date()}")
print(f"Horyzont: {FORECAST_WEEKS} tygodni")
print(f"Liczba polskich kategorii do prognozowania: {feature_df['category_pl'].nunique()}")


In [ ]:
# ── Predykcja rekursywna per kategoria ──
def recursive_forecast(category, n_weeks=4):
    """
    Prognozuje n_weeks tygodni do przodu dla jednej kategorii.
    Każda predykcja staje się danymi historycznymi dla następnej.
    """
    # Pobierz historyczne dane kategorii
    cat_history = feature_df[
        feature_df['category_pl'] == category
    ].sort_values('purchase_week_start').copy()
    
    if len(cat_history) < 8:
        return None  # za mało danych
    
    forecasts = []
    
    for week_ahead in range(1, n_weeks + 1):
        # Ostatni wiersz historii (źródło lagów)
        last_row = cat_history.iloc[-1]
        
        # Następna data
        next_date = last_row['purchase_week_start'] + pd.Timedelta(weeks=1)
        
        # Buduj cechy dla następnego tygodnia
        new_row = {
            'purchase_week_start': next_date,
            'category_pl': category,
            # Lagi units_sold (z historii)
            'lag_1': cat_history.iloc[-1]['units_sold'],
            'lag_2': cat_history.iloc[-2]['units_sold'] if len(cat_history) >= 2 else 0,
            'lag_4': cat_history.iloc[-4]['units_sold'] if len(cat_history) >= 4 else 0,
            'lag_8': cat_history.iloc[-8]['units_sold'] if len(cat_history) >= 8 else 0,
            # Lagi revenue/orders
            'revenue_lag_1': cat_history.iloc[-1]['revenue'],
            'orders_lag_1': cat_history.iloc[-1]['unique_orders'],
            'avg_price_lag_1': cat_history.iloc[-1]['avg_price'],
            # Rolling
            'rolling_mean_4': cat_history['units_sold'].iloc[-4:].mean(),
            'rolling_std_4': cat_history['units_sold'].iloc[-4:].std(),
            'rolling_mean_8': cat_history['units_sold'].iloc[-8:].mean(),
            'rolling_std_8': cat_history['units_sold'].iloc[-8:].std(),
            'revenue_roll_mean_4': cat_history['revenue'].iloc[-4:].mean(),
            'revenue_roll_std_4': cat_history['revenue'].iloc[-4:].std(),
            # Bazowe — bierzemy ostatnie znane
            'avg_price': last_row['avg_price'],
            'avg_freight': last_row['avg_freight'],
            # Czas
            'year': next_date.year,
            'month': next_date.month,
            'quarter': next_date.quarter,
            'week_of_year': next_date.isocalendar()[1],
            'is_month_start': int(next_date.is_month_start),
            'is_month_end': int(next_date.is_month_end),
            'is_holiday': is_holiday_week(next_date),
            # Kategoria
            'category_enc': last_row['category_enc'],
            'category_mean_units': last_row['category_mean_units'],
        }
        
        # Momentum (po obliczeniu rolling)
        new_row['momentum_1_4'] = new_row['lag_1'] - new_row['rolling_mean_4']
        new_row['momentum_1_8'] = new_row['lag_1'] - new_row['rolling_mean_8']
        
        # Pct change
        prev_units = cat_history.iloc[-2]['units_sold'] if len(cat_history) >= 2 else 1
        new_row['units_pct_change_1'] = (
            (new_row['lag_1'] - prev_units) / prev_units if prev_units != 0 else 0
        )
        prev_revenue = cat_history.iloc[-2]['revenue'] if len(cat_history) >= 2 else 1
        new_row['revenue_pct_change_1'] = (
            (new_row['revenue_lag_1'] - prev_revenue) / prev_revenue if prev_revenue != 0 else 0
        )
        
        # Predykcja
        X_pred = pd.DataFrame([new_row])[FEATURES].fillna(0).replace([np.inf, -np.inf], 0)
        pred = max(best_model.predict(X_pred)[0], 0)
        
        forecasts.append({
            'category': category,
            'week_start': next_date,
            'week_ahead': week_ahead,
            'forecast': pred
        })
        
        # Dodaj predykcję jako "history" dla kolejnej iteracji
        new_row['units_sold'] = pred
        new_row['revenue'] = pred * new_row['avg_price']
        new_row['unique_orders'] = pred  # przybliżenie
        cat_history = pd.concat([cat_history, pd.DataFrame([new_row])], ignore_index=True)
    
    return forecasts

# ── Prognoza dla wszystkich kategorii ──
all_categories_list = feature_df['category_pl'].unique()
all_forecasts = []

for cat in all_categories_list:
    fc = recursive_forecast(cat, FORECAST_WEEKS)
    if fc:
        all_forecasts.extend(fc)

forecast_df = pd.DataFrame(all_forecasts)
print(f"Prognoza wygenerowana dla {forecast_df['category'].nunique()} kategorii")
print(f"Łącznie wierszy: {len(forecast_df)}")
print(f"\nPrzykład:")
print(forecast_df.head(8).to_string(index=False))


### Krok 2: Wykrywanie trendów

**Dwa podejścia:**

1. **Prosta metoda:** porównanie sumy ostatnich 4 tyg vs poprzednich 4 tyg
   - `>+10%` → rosnący 📈
   - `<-10%` → spadający 📉
   - inaczej → stabilny ➡️

2. **SARIMA** (Seasonal ARIMA) — model statystyczny szeregu czasowego
   - Wykrywa trend długoterminowy + sezonowość
   - Daje również interwały ufności


In [ ]:
# ── METODA 1: Porównanie 4 tygodni ──
def detect_trend_simple(category):
    """Porównuje sumę ostatnich 4 tyg vs poprzednich 4 tyg."""
    cat_data = feature_df[
        feature_df['category_pl'] == category
    ].sort_values('purchase_week_start').tail(8)
    
    if len(cat_data) < 8:
        return {'category': category, 'trend': 'za mało danych', 'change_pct': np.nan}
    
    last_4 = cat_data['units_sold'].iloc[-4:].sum()
    prev_4 = cat_data['units_sold'].iloc[-8:-4].sum()
    
    if prev_4 == 0:
        change_pct = 0 if last_4 == 0 else 100
    else:
        change_pct = (last_4 - prev_4) / prev_4 * 100
    
    if change_pct > 10:
        trend = 'rosnący 📈'
    elif change_pct < -10:
        trend = 'spadający 📉'
    else:
        trend = 'stabilny ➡️'
    
    return {
        'category': category,
        'last_4w_sales': last_4,
        'prev_4w_sales': prev_4,
        'change_pct': round(change_pct, 1),
        'trend': trend
    }

trends = [detect_trend_simple(cat) for cat in all_categories_list]
trends_df = pd.DataFrame(trends).sort_values('change_pct', ascending=False)

print("=== TOP 10 ROSNĄCYCH KATEGORII ===")
print(trends_df.head(10)[['category', 'last_4w_sales', 'prev_4w_sales', 'change_pct', 'trend']].to_string(index=False))

print("\n=== TOP 10 SPADAJĄCYCH KATEGORII ===")
print(trends_df.tail(10)[['category', 'last_4w_sales', 'prev_4w_sales', 'change_pct', 'trend']].to_string(index=False))


In [ ]:
# ── Podsumowanie trendów ──
trend_summary = trends_df['trend'].value_counts()
print("=== PODSUMOWANIE TRENDÓW ===")
print(trend_summary)

plt.figure(figsize=(10, 5))
colors_map = {'rosnący 📈': '#2ecc71', 'stabilny ➡️': '#3498db', 'spadający 📉': '#e74c3c', 'za mało danych': '#95a5a6'}
trend_summary.plot(kind='bar', color=[colors_map.get(t, '#95a5a6') for t in trend_summary.index])
plt.title('Liczba kategorii w każdej kategorii trendu')
plt.xticks(rotation=0)
plt.ylabel('Liczba kategorii')
plt.tight_layout()
plt.show()


In [ ]:
# ── Top rosnące — wykres ──
top_growing = trends_df.head(10)
top_falling = trends_df.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top_growing['category'], top_growing['change_pct'], color='#2ecc71')
axes[0].set_title('Top 10 rosnących kategorii (% zmiana 4tyg vs poprzednie 4tyg)')
axes[0].set_xlabel('Zmiana %')
axes[0].invert_yaxis()

axes[1].barh(top_falling['category'], top_falling['change_pct'], color='#e74c3c')
axes[1].set_title('Top 10 spadających kategorii')
axes[1].set_xlabel('Zmiana %')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


### SARIMA — model statystyczny

SARIMA (Seasonal ARIMA) bardziej wyrafinowanie wykrywa trend i sezonowość. 
Trenujemy go na 5 największych kategoriach (każda wymaga osobnego modelu, więc liczy się długo).

**Parametry SARIMA(p,d,q)(P,D,Q,s):**
- `p,d,q` — komponent ARIMA (autoregresja, różnicowanie, średnia ruchoma)
- `P,D,Q` — komponenty sezonowe
- `s` — okres sezonowości (52 tygodnie = rok)


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

# ── SARIMA dla TOP 5 kategorii ──
top_5_cats = (
    feature_df.groupby('category_pl')['units_sold']
    .sum().nlargest(5).index.tolist()
)

print(f"Trenowanie SARIMA dla {len(top_5_cats)} top kategorii...")

sarima_results = {}

for cat in top_5_cats:
    cat_ts = (
        feature_df[feature_df['category_pl'] == cat]
        .sort_values('purchase_week_start')
        .set_index('purchase_week_start')['units_sold']
    )
    
    try:
        # SARIMA(1,1,1)(1,1,1,52) — klasyczna konfiguracja dla danych tygodniowych z roczną sezonowością
        # Dla mniejszych danych redukujemy sezonowość
        s = 52 if len(cat_ts) >= 104 else 4
        
        model = SARIMAX(
            cat_ts,
            order=(1, 1, 1),
            seasonal_order=(1, 1, 1, s),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fit = model.fit(disp=False, maxiter=50)
        
        # Prognoza 4 tyg
        forecast = fit.get_forecast(steps=FORECAST_WEEKS)
        pred_mean = np.maximum(forecast.predicted_mean.values, 0)
        pred_ci = forecast.conf_int().values
        pred_ci = np.maximum(pred_ci, 0)
        
        sarima_results[cat] = {
            'history': cat_ts,
            'forecast': pred_mean,
            'ci_lower': pred_ci[:, 0],
            'ci_upper': pred_ci[:, 1],
            'aic': fit.aic
        }
        print(f"  ✓ {cat:30s} AIC: {fit.aic:.1f}")
    except Exception as e:
        print(f"  ✗ {cat}: błąd ({str(e)[:50]})")




In [ ]:
# ── Wizualizacja SARIMA ──
fig, axes = plt.subplots(len(sarima_results), 1, figsize=(16, 4 * len(sarima_results)))
if len(sarima_results) == 1:
    axes = [axes]

for ax, (cat, data) in zip(axes, sarima_results.items()):
    history = data['history'].iloc[-26:]  # ostatnie 26 tyg dla czytelności
    forecast_dates = pd.date_range(
        start=history.index[-1] + pd.Timedelta(weeks=1),
        periods=FORECAST_WEEKS, freq='W-MON'
    )
    
    ax.plot(history.index, history.values, 'b-', label='Historia (6 mies)', linewidth=2)
    ax.plot(forecast_dates, data['forecast'], 'r--', marker='o', label='SARIMA forecast', linewidth=2)
    ax.fill_between(
        forecast_dates,
        data['ci_lower'],
        data['ci_upper'],
        alpha=0.2, color='red', label='95% CI'
    )
    ax.axvline(history.index[-1], color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'{cat} — prognoza SARIMA (4 tygodnie)')
    ax.set_ylabel('Sztuki')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Krok 3: Stany magazynowe — hybrydowy Safety Stock

Liczymy Safety Stock metodą **hybrydową** opartą na RMSE modelu i niepewności czasu dostawy:

```
Safety Stock = Z × √(LT_avg × RMSE² + D_avg² × LT_std²)
```

#### Co oznaczają zmienne:
- **Z = 1.65** — wartość krytyczna dla 95% service level
- **LT_avg, LT_std** — średnia i odchylenie czasu dostawy (z prawdziwych danych Olist, kolumna `delivery_days`)
- **RMSE** — błąd naszego modelu **per kategoria** (jak bardzo XGBoost myli się dla danej kategorii na test set)
- **D_avg** — średnia prognoza tygodniowa z modelu

#### Dlaczego to podejście, a nie klasyczne `Z × σ × √(LT)`?

Klasyczna metoda zakłada, że ryzyko = historyczne wahania popytu (`σ_demand`). Ale my **mamy model**, który już wyłapał regularne wzorce: sezonowość, święta, trendy, lagi. To wszystko jest "wykorzystane" przez model. **Zostaje tylko ryzyko residualne** — jak bardzo model się myli mimo treningu = **RMSE**.

Plus dochodzi drugi czynnik: **dostawcy się spóźniają**. Klasyczna metoda to ignoruje, hybrydowa uwzględnia (`σ_LT`).

#### Reorder Point
```
Reorder Point = (D_avg × LT_avg) + Safety_Stock
```
Poziom zapasu, przy którym należy złożyć kolejne zamówienie u dostawcy.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OBLICZENIA STANÓW MAGAZYNOWYCH — HYBRYDOWY SAFETY STOCK
# ═══════════════════════════════════════════════════════════════════

SERVICE_LEVEL_Z = 1.65   # 95% service level (dla 99% użyj 2.33)

# ── Lead Time z prawdziwych danych dostawy ──
lt_data = delivered_df['delivery_days'].dropna()
lt_data = lt_data[(lt_data >= 0) & (lt_data <= 60)]  # filtrujemy outliery

LT_AVG_DAYS = lt_data.mean()
LT_STD_DAYS = lt_data.std()
LT_AVG_WEEKS = LT_AVG_DAYS / 7
LT_STD_WEEKS = LT_STD_DAYS / 7

print("=" * 60)
print("PARAMETRY LEAD TIME (z prawdziwych danych Olist)")
print("=" * 60)
print(f"  Średni czas dostawy:     {LT_AVG_DAYS:.1f} dni  ({LT_AVG_WEEKS:.2f} tyg)")
print(f"  Odchylenie czasu dost.:  {LT_STD_DAYS:.1f} dni  ({LT_STD_WEEKS:.2f} tyg)")
print()

# ── RMSE per kategoria (z modelu XGBoost tuned na test set) ──
test_with_pred = feature_df.loc[test_mask].copy()
test_with_pred['pred'] = np.maximum(best_model.predict(X_test), 0)

category_rmse = test_with_pred.groupby('category_pl').apply(
    lambda x: np.sqrt(np.mean((x['units_sold'] - x['pred']) ** 2))
).reset_index(name='model_rmse')

# Globalny RMSE jako fallback dla kategorii bez danych w teście
global_rmse = np.sqrt(np.mean((test_with_pred['units_sold'] - test_with_pred['pred']) ** 2))
print(f"  Globalny RMSE modelu:    {global_rmse:.2f} sztuk")
print(f"\n  RMSE per kategoria — top 5 najtrudniejszych do prognozowania:")
for _, row in category_rmse.nlargest(5, 'model_rmse').iterrows():
    print(f"    {row['category_pl']:30s} RMSE: {row['model_rmse']:.2f}")
print()

# ── Główna pętla: liczymy hybrydowy safety stock per kategoria ──
inventory_recs = []

for cat in all_categories_list:
    cat_forecast = forecast_df[forecast_df['category'] == cat]
    if len(cat_forecast) == 0:
        continue
    
    cat_history = feature_df[
        feature_df['category_pl'] == cat
    ].sort_values('purchase_week_start')
    
    if len(cat_history) < 8:
        continue
    
    # Statystyki prognoz
    forecast_total = cat_forecast['forecast'].sum()
    forecast_avg = cat_forecast['forecast'].mean()
    
    # RMSE modelu per kategoria (fallback do globalnego jeśli brak)
    rmse_row = category_rmse[category_rmse['category_pl'] == cat]
    cat_model_rmse = rmse_row['model_rmse'].values[0] if len(rmse_row) > 0 else global_rmse
    
    # ── HYBRYDOWY SAFETY STOCK ──
    safety_stock = SERVICE_LEVEL_Z * np.sqrt(
        LT_AVG_WEEKS * (cat_model_rmse ** 2) +
        (forecast_avg ** 2) * (LT_STD_WEEKS ** 2)
    )
    safety_stock = np.ceil(safety_stock)
    
    reorder_point = np.ceil(forecast_avg * LT_AVG_WEEKS + safety_stock)
    recommended_stock = np.ceil(forecast_total + safety_stock)
    
    # Trend
    trend_info = trends_df[trends_df['category'] == cat]
    trend_label = trend_info['trend'].iloc[0] if len(trend_info) > 0 else 'b/d'
    change_pct = trend_info['change_pct'].iloc[0] if len(trend_info) > 0 else 0
    
    inventory_recs.append({
        'category': cat,
        # Prognoza tygodniowa
        'forecast_week_1': round(cat_forecast.iloc[0]['forecast'], 1),
        'forecast_week_2': round(cat_forecast.iloc[1]['forecast'], 1),
        'forecast_week_3': round(cat_forecast.iloc[2]['forecast'], 1),
        'forecast_week_4': round(cat_forecast.iloc[3]['forecast'], 1),
        'forecast_total_4w': round(forecast_total, 1),
        'avg_weekly_demand': round(forecast_avg, 1),
        # Parametry ryzyka
        'model_rmse': round(cat_model_rmse, 1),
        # Stany magazynowe
        'safety_stock': safety_stock,
        'reorder_point': reorder_point,
        'recommended_stock_4w': recommended_stock,
        # Trend
        'trend': trend_label,
        'change_pct': change_pct
    })

inventory_df = pd.DataFrame(inventory_recs).sort_values('forecast_total_4w', ascending=False)

print("=" * 60)
print("TOP 15 KATEGORII WG ZAPOTRZEBOWANIA")
print("=" * 60)
display_cols = ['category', 'avg_weekly_demand', 'forecast_total_4w',
                'model_rmse', 'safety_stock', 'reorder_point',
                'recommended_stock_4w', 'trend']
print(inventory_df[display_cols].head(15).to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# WIZUALIZACJA: STRUKTURA ZAPASU (popyt + safety stock)
# ═══════════════════════════════════════════════════════════════════

top_10 = inventory_df.head(10).copy()
top_10['avg_plus_ss'] = top_10['avg_weekly_demand'] + top_10['safety_stock']

plt.figure(figsize=(14, 8))
sns.set_theme(style='whitegrid')

# Najpierw rysujemy całkowitą sumę (popyt + SS) — to da nam "narzut SS"
sns.barplot(data=top_10, x='avg_plus_ss', y='category',
            color='lightcoral', label='Safety Stock (narzut)')

# Na to nakładamy bazowy popyt
sns.barplot(data=top_10, x='avg_weekly_demand', y='category',
            color='steelblue', label='Średni popyt tygodniowy')

plt.title('Struktura Zapasu: Popyt bazowy vs Safety Stock (Top 10 kategorii)', fontsize=14)
plt.xlabel('Sztuki')
plt.ylabel('Kategoria (PL)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Dodatkowy wykres: rozkład safety stock
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(inventory_df['safety_stock'], bins=30, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[0].axvline(inventory_df['safety_stock'].median(), color='blue', linestyle='--',
                label=f"Mediana: {inventory_df['safety_stock'].median():.0f}")
axes[0].set_title('Rozkład Safety Stock w kategoriach')
axes[0].set_xlabel('Safety Stock (sztuki)')
axes[0].set_ylabel('Liczba kategorii')
axes[0].legend()

# Top 15 z największym safety stock (najtrudniejsze do prognozowania)
top_ss = inventory_df.nlargest(15, 'safety_stock')[['category', 'safety_stock', 'model_rmse']]
sns.barplot(data=top_ss, x='safety_stock', y='category', palette='Reds_r', ax=axes[1])
axes[1].set_title('Top 15 kategorii z największym Safety Stock\n(= najtrudniejsze do prognozowania)')
axes[1].set_xlabel('Safety Stock (sztuki)')

plt.tight_layout()
plt.show()


### Interpretacja biznesowa metryk modelu

Tłumaczymy nasze metryki RMSE/MAE/MAPE językiem zrozumiałym dla działu zakupów:

- **MAE ≈ 7.6 sztuk** — średnio model myli się o ~8 sztuk na kategorię/tydzień. To bardzo namacalna wartość: o tyle sztuk będzie w magazynie za dużo lub za mało w stosunku do idealnej prognozy.

- **RMSE ≈ 16.3 sztuk** — RMSE mocniej karze model za duże pomyłki (rzadkie sytuacje, gdy mocno się mylimy). **Używamy go bezpośrednio do liczenia Safety Stock w metodzie hybrydowej.** Wysokie RMSE = popyt skokowy = większy bufor.

- **MAPE ≈ 100%** — błąd procentowy. Wygląda groźnie, ale to artefakt tego, że niektóre kategorie sprzedają się 1-2 sztuki/tydzień. Pomyłka o 1 sztukę przy 1 sprzedanej = MAPE 100%, choć w sztukach (MAE) to nic.

**Wniosek:** dla zarządzania magazynem **RMSE i MAE** są najważniejsze, bo liczy się fizyczne miejsce na półce, a nie procenty.

#### Dlaczego hybrydowa metoda jest lepsza?

Klasyczna metoda zakłada że ryzyko = historyczne wahania popytu. Ale my **mamy model**, który już wyłapał regularne wzorce (sezonowość, święta, trendy). Zostaje tylko **ryzyko residualne** — jak bardzo model się myli mimo nauki = **RMSE**.

Plus: dostawcy się spóźniają (`σ_LT`). Klasyczna metoda to ignoruje, hybrydowa uwzględnia.


### Krok 4: Eksport do CSV

Generujemy 3 pliki CSV dla działu logistyki:

1. **`forecast_4_weeks.csv`** — szczegółowa prognoza per kategoria × tydzień
2. **`inventory_recommendations.csv`** — rekomendacje stanów magazynowych
3. **`trends_summary.csv`** — analiza trendów wszystkich kategorii


In [ ]:
# ── Eksport CSV ──
import os
from pathlib import Path

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# 1. Prognoza per kategoria × tydzień
forecast_export = forecast_df.copy()
forecast_export['forecast'] = forecast_export['forecast'].round(1)
forecast_export.to_csv(output_dir / 'forecast_4_weeks.csv', index=False, encoding='utf-8')

# 2. Rekomendacje stanów magazynowych — ZAWIERA OBIE METODY
inventory_df.to_csv(output_dir / 'inventory_recommendations.csv', index=False, encoding='utf-8')

# 3. Trendy
trends_df.to_csv(output_dir / 'trends_summary.csv', index=False, encoding='utf-8')

print(f"Pliki zapisane w folderze: {output_dir.absolute()}/")
print(f"  1. forecast_4_weeks.csv          ({len(forecast_export)} wierszy)")
print(f"  2. inventory_recommendations.csv ({len(inventory_df)} wierszy, obie metody SS)")
print(f"  3. trends_summary.csv            ({len(trends_df)} wierszy)")


In [ ]:
# ── Podgląd plików CSV ──
print("=== forecast_4_weeks.csv ===")
print(forecast_export.head().to_string(index=False))

print("\n=== inventory_recommendations.csv ===")
print(inventory_df.head().to_string(index=False))

print("\n=== trends_summary.csv ===")
print(trends_df.head().to_string(index=False))


## Etap 5: Aplikacja webowa

📌 Udostępnienie modelu i wyników w formie aplikacji:
- Integracja modelu
- Dashboard (wykresy, metryki)
- API / Streamlit / Flask
- Interakcja użytkownika
